In [1]:
# ===== Cell 1: Imports + Config =====
import os, json, re, gc
import torch, pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

MODEL_NAME    = '/root/autodl-tmp/employee_code/base_model'
TRAIN_JSON    = '/root/autodl-tmp/cpl_ft/data/train_sft_cpl.json'
EVAL_JSON     = '/root/autodl-tmp/cpl_ft/data/eval_cpl_questions.json'
OUTPUT_DIR    = '/root/autodl-tmp/cpl_ft/checkpoints_v2'
RESULTS_DIR   = '/root/autodl-tmp/cpl_ft/results_v2'

MAX_SEQ_LEN   = 1024
LORA_R = 32; LORA_ALPHA = 64; LORA_DROPOUT = 0.05
LR = 1e-4; EPOCHS = 5.0; BS = 2; GAS = 8
SAVE_STEPS = 10; LOG_STEPS = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'920 samples / 16 eff * 5 epochs = ~290 steps')
print(f'Save every {SAVE_STEPS} steps -> ~29 checkpoints')
print(f'Eval: only 150-290 range')

920 samples / 16 eff * 5 epochs = ~290 steps
Save every 10 steps -> ~29 checkpoints
Eval: only 150-290 range


In [2]:
# ===== Cell 2: Tokenizer + Model + Data + Train + Save =====
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', trust_remote_code=True)
print(f'CUDA: {torch.cuda.is_available()}')

# Dataset
SYS = '/no_think\n你是一个专业的中国法律背诵助手。请严格根据用户要求输出法条内容，不要添加解释。'
with open(TRAIN_JSON, 'r', encoding='utf-8') as f: raw = json.load(f)
examples = []
for item in raw:
    inst = item.get('instruction','').strip()
    inp  = item.get('input','').strip()
    out  = item.get('output','').strip()
    if not inst or not out: continue
    msgs = [{'role':'system','content':SYS},{'role':'user','content':inst if not inp else f'{inst}\n\n{inp}'}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    examples.append({'prompt':prompt, 'completion':out + tokenizer.eos_token})
train_ds = Dataset.from_list(examples)
print(f'Train: {len(train_ds)} samples')

# Train
peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','v_proj','o_proj'])
training_args = SFTConfig(output_dir=OUTPUT_DIR, learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BS, gradient_accumulation_steps=GAS,
    logging_steps=LOG_STEPS, save_steps=SAVE_STEPS, save_strategy='steps',
    save_only_model=True,
    bf16=torch.cuda.is_available(), fp16=not torch.cuda.is_available(),
    max_length=MAX_SEQ_LEN, packing=False, report_to='none',
    warmup_ratio=0.03, lr_scheduler_type='cosine')
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_ds,
    processing_class=tokenizer, peft_config=peft_config)
trainer.train()

# Save final
fd = os.path.join(OUTPUT_DIR, 'final_adapter')
trainer.model.save_pretrained(fd); tokenizer.save_pretrained(fd)
print(f'final_adapter: {fd}')
del trainer; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


CUDA: True
Train: 920 samples


Adding EOS to train dataset:   0%|          | 0/920 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/920 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.209551
20,0.517500
30,0.397626
40,0.375891
50,0.333254
60,0.328474
70,0.208360
80,0.238692
90,0.240936
100,0.227007


final_adapter: /root/autodl-tmp/cpl_ft/checkpoints_v2/final_adapter


In [3]:
# ===== Cell 3: Eval functions =====
TOP5_PROMPT = '你是一个专业的中国法律检索专家。\n\n你的任务不是作答，而是为后续数据库检索生成候选法条。\n请根据用户案情，输出最相关的5条法条编号，按相关性从高到低排序。\n\n硬性要求：\n1. 必须恰好输出5条；\n2. 每条都必须是《法律名称》第XXX条；\n3. 不要输出解释、分析、理由、序号或其他任何文字；\n4. 即使不确定，也必须给出5条最可能的候选法条；\n5. 优先输出最核心、最直接适用的法条；\n6. 只输出 JSON，不要输出其他内容。\n\n输出格式：\n{"articles": ["《法律名称》第XXX条", ...]}'
ART_RE = r'《.+?》第[0-9０-９一二三四五六七八九十百千万零〇○两]+条'
CN_M = {'零':0,'〇':0,'○':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
CN_U = {'十':10,'百':100,'千':1000,'万':10000}

def c2i(t):
    if not t: return None
    t=str(t).strip().translate(str.maketrans('０１２３４５６７８９','0123456789'))
    if t.isdigit(): return int(t)
    total=section=number=0
    for ch in t:
        if ch in CN_M: number=CN_M[ch]
        elif ch in CN_U:
            u=CN_U[ch]
            if u==10000: section=(section+(number or 0))*u; total+=section; section=number=0
            else:
                if number==0: number=1
                section+=number*u; number=0
        else: return None
    return total+section+number

def canon(art):
    if not isinstance(art,str): return ''
    x=art.strip(); x=re.sub(r'\s+','',x); x=x.replace('（','(').replace('）',')')
    x=x.translate(str.maketrans('０１２３４５６７８９','0123456789'))
    m=re.match(r'^(《.*?》)第([0-9一二三四五六七八九十百千万零〇○两]+)条$',x)
    if not m: return x
    law=re.sub(r'^《中华人民共和国','《',m.group(1))
    no=c2i(m.group(2))
    return f'{law}第{no}条' if no is not None else f'{law}第{m.group(2)}条'

def extract_articles(text,topk=5):
    mentions=[]
    m=re.search(r'\{.*\}',text,re.S)
    if m:
        try:
            obj=json.loads(m.group())
            if isinstance(obj,dict) and isinstance(obj.get('articles'),list):
                for x in obj['articles']:
                    if isinstance(x,str): mentions.extend(re.findall(ART_RE,x))
        except: pass
    if not mentions: mentions=re.findall(ART_RE,text)[:topk]
    else:
        for m2 in re.findall(ART_RE,text):
            if len(mentions)>=topk: break
            mentions.append(m2)
    return [canon(x) for x in mentions[:topk] if canon(x)]

@torch.no_grad()
def generate_top5(question, mdl):
    msgs=[{'role':'system','content':TOP5_PROMPT},
          {'role':'user','content':f'/no_think\n案情：\n{question}\n\n请严格输出 JSON。'}]
    text=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=tokenizer(text,return_tensors='pt').to(mdl.device)
    outputs=mdl.generate(**inputs,max_new_tokens=256,do_sample=True,repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,pad_token_id=tokenizer.pad_token_id)
    raw=tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
    return {'raw':raw,'pred':extract_articles(raw)}

def eval_one(adapter_path, label):
    gc.collect(); torch.cuda.empty_cache()
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True)
    mdl = PeftModel.from_pretrained(base, adapter_path, is_trainable=False); mdl.eval()
    results=[]; tp_total=pred_total=gt_total=0
    for item in tqdm(eval_data, desc=label):
        gen = generate_top5(item['question'], mdl)
        ps=set(gen['pred']); gs=set(canon(a) for a in item['articles'] if canon(a))
        tp=len(ps&gs); p=tp/max(len(ps),1); r=tp/max(len(gs),1)
        f1=2*p*r/(p+r) if p+r>0 else 0.0
        tp_total+=tp; pred_total+=len(ps); gt_total+=len(gs)
        results.append({'id':item['id'],'question':item['question'],'gold':list(gs),'pred':gen['pred'],
            'P':round(p,6),'R':round(r,6),'F1':round(f1,6),'raw':gen['raw']})
    mp=sum(r['P'] for r in results)/len(results)
    mr=sum(r['R'] for r in results)/len(results)
    mf1=sum(r['F1'] for r in results)/len(results)
    del mdl, base; gc.collect(); torch.cuda.empty_cache()
    s = int(label.split('-')[-1]) if '-' in label else 0
    return {'label':label,'step':s,'P@5':round(mp,4),'R@5':round(mr,4),'F1@5':round(mf1,4),'per_sample':results}

In [4]:
# ===== Cell 4: Eval 150-290 + Leaderboard =====
with open(EVAL_JSON, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)
print(f'Eval questions: {len(eval_data)}')

targets = []
for name in sorted(os.listdir(OUTPUT_DIR)):
    full = os.path.join(OUTPUT_DIR, name)
    if os.path.isdir(full) and name.startswith('checkpoint-'):
        s = name.split('-')[-1]
        if s.isdigit():
            step = int(s)
            if 150 <= step <= 290:
                targets.append((step, name, full))
targets = sorted(targets, key=lambda x: x[0])
final_ckpt = os.path.join(OUTPUT_DIR, 'final_adapter')
if os.path.isdir(final_ckpt):
    targets.append((9999, 'final_adapter', final_ckpt))
print(f'Checkpoints (150-290): {len(targets)}')
for step, name, _ in targets:
    print(f'  {name} (step {step})')

all_rows = []
for step, label, adapter_path in targets:
    print(f'\n--- {label} ---')
    row = eval_one(adapter_path, label)
    print(f'P@5={row["P@5"]:.4f} R@5={row["R@5"]:.4f} F1@5={row["F1@5"]:.4f}')
    all_rows.append({'Checkpoint':label,'Step':step,'P@5':row['P@5'],'R@5':row['R@5'],'F1@5':row['F1@5']})
    with open(os.path.join(RESULTS_DIR, f'{label}.json'), 'w', encoding='utf-8') as f:
        json.dump(row, f, ensure_ascii=False, indent=2)

print('\n' + '=' * 60)
print('LEADERBOARD (150-290)')
print('=' * 60)
df = pd.DataFrame(all_rows).sort_values('F1@5', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))
best = df.iloc[0]
print(f'\n*** BEST: {best["Checkpoint"]} F1@5={best["F1@5"]:.4f} ***')

with open(os.path.join(RESULTS_DIR, 'leaderboard.json'), 'w', encoding='utf-8') as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)
print(f'Saved: {RESULTS_DIR}/')

Eval questions: 48
Checkpoints (150-290): 16
  checkpoint-150 (step 150)
  checkpoint-160 (step 160)
  checkpoint-170 (step 170)
  checkpoint-180 (step 180)
  checkpoint-190 (step 190)
  checkpoint-200 (step 200)
  checkpoint-210 (step 210)
  checkpoint-220 (step 220)
  checkpoint-230 (step 230)
  checkpoint-240 (step 240)
  checkpoint-250 (step 250)
  checkpoint-260 (step 260)
  checkpoint-270 (step 270)
  checkpoint-280 (step 280)
  checkpoint-290 (step 290)
  final_adapter (step 9999)

--- checkpoint-150 ---


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

checkpoint-150:   0%|          | 0/48 [00:00<?, ?it/s]

P@5=0.0844 R@5=0.3854 F1@5=0.1373

--- checkpoint-160 ---


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

checkpoint-290:   0%|          | 0/48 [00:00<?, ?it/s]

P@5=0.0792 R@5=0.3438 F1@5=0.1270

--- final_adapter ---


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

P@5=0.0708 R@5=0.2812 F1@5=0.1111

LEADERBOARD (150-290)
    Checkpoint  Step    P@5    R@5   F1@5
checkpoint-180   180 0.0969 0.4167 0.1552
checkpoint-200   200 0.0944 0.3958 0.1503
checkpoint-240   240 0.0917 0.4062 0.1478
checkpoint-230   230 0.0917 0.4062 0.1478
checkpoint-220   220 0.0917 0.3958 0.1468
checkpoint-260   260 0.0854 0.3750 0.1377
checkpoint-150   150 0.0844 0.3854 0.1373
checkpoint-170   170 0.0792 0.3438 0.1270
checkpoint-290   290 0.0792 0.3438 0.1270
checkpoint-250   250 0.0760 0.3229 0.1210
 final_adapter  9999 0.0708 0.2812 0.1111
checkpoint-270   270 0.0667 0.2917 0.1071
checkpoint-190   190 0.0625 0.2604 0.0992
checkpoint-280   280 0.0625 0.2604 0.0992
checkpoint-160   160 0.0583 0.2292 0.0913
checkpoint-210   210 0.0542 0.2188 0.0853

*** BEST: checkpoint-180 F1@5=0.1552 ***
Saved: /root/autodl-tmp/cpl_ft/results_v2/
